# Data-Driven Dispersion Initialization Analysis

Compute per-gene NB dispersion theta from method of moments on raw counts.

Variance decomposition:
- A: Poisson noise = mu_g
- B+C: Library size variability = mu_g² · CV²(L)
- D: Gene-specific NB overdispersion = (mu_g²/θ_g) · (1 + CV²(L))

Theta formula: `θ = mu² / excess_technical`, where
`excess_technical = (Var - mu - mu²·CV²) / ((1+CV²) · bio_frac)`

See `.claude/plans/dispersion-init-plan.md` for full derivation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS_DIR = Path("../../../results/dispersion_init_analysis")

# All 6 datasets
DATASETS = {
    "bm_rna": "Bone marrow RNA (69k cells, 26k genes)",
    "bm_atac": "Bone marrow ATAC (69k cells, 116k peaks)",
    "immune_rna": "Immune RNA (706k cells, 26k genes)",
    "immune_atac": "Immune ATAC tiles (690k cells, 3M tiles)",
    "embryo_rna": "Embryo RNA (627k cells, 37k genes)",
    "embryo_atac": "Embryo ATAC (540k cells, 3M tiles)",
}

BIO_FRACS = [1, 5, 10, 20]
EPS = 1e-10

In [ ]:
# Load precomputed diagnostics from npz files
results = {}
for name in DATASETS:
    npz_path = RESULTS_DIR / f"{name}.npz"
    if npz_path.exists():
        data = dict(np.load(npz_path, allow_pickle=True))
        # Extract scalars from length-1 arrays
        data["cv2_L"] = float(data["cv2_L"][0])
        data["n_sub_poisson"] = int(data["n_sub_poisson"][0])
        data["cv2_L_within_batch"] = float(data["cv2_L_within_batch"][0])
        data["cv2_L_between_batch"] = float(data["cv2_L_between_batch"][0])
        results[name] = data
        print(f"Loaded {name}: {len(data['mean_g'])} genes, CV²(L)={data['cv2_L']:.4f}")
    else:
        print(f"MISSING: {npz_path}")

print(f"\nLoaded {len(results)}/{len(DATASETS)} datasets")


def compute_theta(diag, bio_frac):
    """Recompute theta for a given bio_frac from saved diagnostics."""
    nb_inflation = 1 + diag["cv2_L"]
    excess_tech = diag["excess_raw"] / (nb_inflation * bio_frac)
    return (diag["mean_g"] ** 2) / np.maximum(excess_tech, EPS)

## Quantile Table: All datasets × bio_frac=[1, 5, 10, 20]

In [ ]:
# Per-dataset tables: bio_frac=[1,5,10,20] for both options
# Option 1: full correction (Poisson + library size removed)
#   excess = (Var - mu - mu²·CV²) / (1+CV²)
#   theta = mu² / (excess / bio_frac)
# Option 2: simple (Poisson removed only, library size NOT removed)
#   excess = Var - mu
#   theta = mu² / (excess / bio_frac)


def compute_theta_simple(diag, bio_frac):
    """Option 2: only subtract Poisson, keep library size variance."""
    excess_simple = diag["var_g"] - diag["poisson_var"]
    excess_tech = excess_simple / bio_frac
    return (diag["mean_g"] ** 2) / np.maximum(excess_tech, EPS)


q_pcts = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
q_labels = ["1%", "5%", "10%", "25%", "50%", "75%", "90%", "95%", "99%"]

for name in sorted(results):
    diag = results[name]
    n_genes = len(diag["mean_g"])
    print(f"\n{'=' * 110}")
    print(f"  {name} — {DATASETS[name]}")
    print(f"  CV²(L)={diag['cv2_L']:.4f}, sub-Poisson={diag['n_sub_poisson']}/{n_genes}")
    print(f"{'=' * 110}")

    for option_label, theta_fn in [
        ("Option 1: cell size REMOVED", compute_theta),
        ("Option 2: cell size KEPT", compute_theta_simple),
    ]:
        print(f"\n  {option_label}")
        header = f"  {'bio_frac':>8s} " + " ".join(f"{q:>8s}" for q in q_labels)
        print(header)
        print(f"  {'-' * 100}")
        for bf in BIO_FRACS:
            theta = theta_fn(diag, bf)
            finite_t = theta[np.isfinite(theta) & (theta > 0)]
            qs = np.quantile(finite_t, q_pcts)
            print(f"  {bf:>8.0f} " + " ".join(f"{v:>8.3f}" for v in qs))

In [ ]:
# Cross-dataset summary: median theta for each bio_frac, both options
for option_label, theta_fn in [
    ("Option 1: cell size REMOVED (Poisson + library subtracted)", compute_theta),
    ("Option 2: cell size KEPT (Poisson only subtracted)", compute_theta_simple),
]:
    print(f"\n{'=' * 80}")
    print(f"  {option_label}")
    print(f"{'=' * 80}")
    print(f"  {'Dataset':<16s} {'CV²(L)':>7s} {'n_genes':>8s}", end="")
    for bf in BIO_FRACS:
        print(f" {'bf=' + str(bf):>10s}", end="")
    print()
    print(f"  {'-' * 70}")
    for name in sorted(results):
        diag = results[name]
        n = len(diag["mean_g"])
        print(f"  {name:<16s} {diag['cv2_L']:>7.3f} {n:>8d}", end="")
        for bf in BIO_FRACS:
            theta = theta_fn(diag, bf)
            finite_t = theta[np.isfinite(theta) & (theta > 0)]
            print(f" {np.median(finite_t):>10.3f}", end="")
        print()

## Histograms: log(theta) distribution per dataset (bio_frac=10)

In [ ]:
# Histograms of log(theta) for bio_frac=10
n_ds = len(results)
fig, axes = plt.subplots(1, n_ds, figsize=(5 * n_ds, 4.5), squeeze=False)

for i, (name, diag) in enumerate(sorted(results.items())):
    ax = axes[0, i]
    theta = compute_theta(diag, bio_frac=10)
    log_t = np.log(np.clip(theta, 1e-6, 1e8))
    finite = np.isfinite(log_t)
    ax.hist(log_t[finite], bins=100, alpha=0.7, color="steelblue", edgecolor="none")
    ax.axvline(np.log(1), color="red", ls="--", lw=1.5, label="θ=1")
    ax.axvline(np.log(9), color="orange", ls="--", lw=1.5, label="θ=9 (default)")
    ax.axvline(np.log(0.3), color="purple", ls="--", lw=1.5, label="θ=0.3")
    ax.set_title(f"{name}\nCV²(L)={diag['cv2_L']:.3f}, subPoi={diag['n_sub_poisson']}", fontsize=9)
    ax.set_xlabel("log(θ)")
    ax.set_ylabel("# genes")
    ax.legend(fontsize=6)

plt.suptitle("log(θ) distribution — bio_frac=10", fontsize=12)
plt.tight_layout()
plt.show()

## hist2d: theta vs gene mean (mean-dispersion relationship, bio_frac=10)

In [ ]:
# hist2d: log10(theta) vs log10(mean) for each dataset
n_ds = len(results)
fig, axes = plt.subplots(2, n_ds, figsize=(5 * n_ds, 9), squeeze=False)

for i, (name, diag) in enumerate(sorted(results.items())):
    theta = compute_theta(diag, bio_frac=10)
    mean_g = diag["mean_g"]
    valid = (theta > 1e-4) & (theta < 1e6) & (mean_g > 0)

    # Log-log
    ax = axes[0, i]
    ax.hist2d(np.log10(mean_g[valid]), np.log10(theta[valid]), bins=100, cmap="viridis", cmin=1)
    ax.axhline(0, color="red", ls="--", alpha=0.7, label="θ=1")
    ax.axhline(np.log10(9), color="orange", ls="--", alpha=0.7, label="θ=9")
    ax.set_xlabel("log10(mean count)")
    ax.set_ylabel("log10(θ)")
    ax.set_title(f"{name} — log-log", fontsize=9)
    ax.legend(fontsize=6)

    # Linear (clipped)
    ax = axes[1, i]
    valid2 = valid & (theta < 50) & (mean_g < 50)
    if valid2.sum() > 10:
        ax.hist2d(mean_g[valid2], theta[valid2], bins=100, cmap="viridis", cmin=1)
    ax.axhline(1, color="red", ls="--", alpha=0.7, label="θ=1")
    ax.axhline(9, color="orange", ls="--", alpha=0.7, label="θ=9")
    ax.set_xlabel("mean count")
    ax.set_ylabel("θ")
    ax.set_title(f"{name} — linear (<50)", fontsize=9)
    ax.legend(fontsize=6)

plt.suptitle("Mean-dispersion relationship — bio_frac=10", fontsize=12)
plt.tight_layout()
plt.show()

## Variance decomposition: fraction of variance explained

In [ ]:
n_ds = len(results)
fig, axes = plt.subplots(1, n_ds, figsize=(4 * n_ds, 4.5), squeeze=False)

for i, (name, diag) in enumerate(sorted(results.items())):
    ax = axes[0, i]
    total = diag["var_g"]
    poisson = diag["poisson_var"]
    library = diag["library_var"]
    excess = diag["excess_raw"]

    valid = total > 0
    frac_poisson = np.median(poisson[valid] / total[valid])
    frac_library = np.median(library[valid] / total[valid])
    frac_excess = np.median(np.maximum(excess[valid], 0) / total[valid])

    bars = ax.bar(["Poisson", "Library", "Gene-\nspecific"], [frac_poisson, frac_library, frac_excess])
    ax.set_title(f"{name}", fontsize=9)
    ax.set_ylabel("Median fraction of variance")
    for j, v in enumerate([frac_poisson, frac_library, frac_excess]):
        ax.text(j, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)

plt.suptitle("Variance decomposition", fontsize=12)
plt.tight_layout()
plt.show()

## Learned theta from P5 experiments

Compare P5 learned theta (init=1, 2, 3) with MoM estimates.

In [ ]:
import torch

RESULTS_BASE = Path("../../../results/")


def load_learned_theta(model_path):
    """Load learned theta from model checkpoint."""
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
    sd = checkpoint["model_state_dict"]
    px_r_mu = sd["px_r_mu"].numpy()
    if px_r_mu.ndim == 2:
        px_r_mu = px_r_mu.mean(axis=1)
    return np.exp(px_r_mu)


p5_models = {
    "default (init=3)": "immune_integration_rna_embryo_es2_no_tea_small",
    "disp1 (init=1)": "immune_integration_rna_embryo_es2_no_tea_small_disp1",
    "disp2 (init=2)": "immune_integration_rna_embryo_es2_no_tea_small_disp2",
}

learned = {}
for label, dirname in p5_models.items():
    path = RESULTS_BASE / dirname / "model" / "model.pt"
    if path.exists():
        learned[label] = load_learned_theta(str(path))
        print(f"{label}: {len(learned[label])} genes, median θ = {np.median(learned[label]):.2f}")
    else:
        print(f"MISSING: {path}")

## hist2d: init=1 vs init=2 vs init=3 (learned theta)

In [ ]:
pairs = [
    ("disp1 (init=1)", "default (init=3)"),
    ("disp2 (init=2)", "default (init=3)"),
    ("disp1 (init=1)", "disp2 (init=2)"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (a, b) in zip(axes, pairs, strict=True):
    ta, tb = learned[a], learned[b]
    n = min(len(ta), len(tb))
    ax.hist2d(np.log10(ta[:n]), np.log10(tb[:n]), bins=100, cmap="viridis", cmin=1)
    lim = [min(np.log10(ta[:n]).min(), np.log10(tb[:n]).min()), max(np.log10(ta[:n]).max(), np.log10(tb[:n]).max())]
    ax.plot(lim, lim, "r--", alpha=0.5)
    ax.set_xlabel(f"log10(θ) — {a}")
    ax.set_ylabel(f"log10(θ) — {b}")
    ax.set_title(f"{a} vs {b}")

plt.tight_layout()
plt.show()

## hist2d: MoM theta vs learned theta (immune RNA, per gene)

In [ ]:
# hist2d: MoM theta vs learned theta for each bio_frac
if "immune_rna" in results and learned:
    for bf in BIO_FRACS:
        mom_theta = compute_theta(results["immune_rna"], bf)

        fig, axes = plt.subplots(1, len(learned), figsize=(6 * len(learned), 5))
        if not hasattr(axes, "__len__"):
            axes = [axes]

        for ax, (label, lt) in zip(axes, learned.items(), strict=False):
            n = min(len(mom_theta), len(lt))
            valid = (mom_theta[:n] > 1e-4) & (mom_theta[:n] < 1e6)
            ax.hist2d(
                np.log10(mom_theta[:n][valid]),
                np.log10(lt[:n][valid]),
                bins=100,
                cmap="viridis",
                cmin=1,
            )
            ax.plot([-2, 5], [-2, 5], "r--", alpha=0.5)
            ax.set_xlabel("log10(θ MoM)")
            ax.set_ylabel(f"log10(θ learned) — {label}")
            ax.set_title(f"{label}")

        plt.suptitle(f"MoM (bio_frac={bf}) vs Learned — immune RNA", fontsize=12)
        plt.tight_layout()
        plt.show()